In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
#import scikitlearn as sk


In [8]:
sessions_df = pd.read_csv('../data/charging_sessions.csv')
weather_df = pd.read_csv('../data/weather_burbank_airport.csv')

sessions_df["connectionTime"] = pd.to_datetime(
    sessions_df["connectionTime"]
)
sessions_df["disconnectTime"] = pd.to_datetime(
    sessions_df["disconnectTime"]
)
sessions_df["doneChargingTime"] = pd.to_datetime(
    sessions_df["doneChargingTime"]
)

# Gesamtdauer in Stunden berechnen
sessions_df["total_duration_hours"] = (
    sessions_df["disconnectTime"] -
    sessions_df["connectionTime"]
).dt.total_seconds() / 3600

# Ungenutzte Zeit in Stunden berechnen
sessions_df["unused_time_hours"] = (
    sessions_df["disconnectTime"] -
    sessions_df["doneChargingTime"]
).dt.total_seconds() / 3600

# Ladedauer in Stunden berechnen
sessions_df["charging_duration_hours"] = (
    sessions_df["doneChargingTime"] -
    sessions_df["connectionTime"]
).dt.total_seconds() / 3600

sessions_df.head()


,Unnamed: 0,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs,total_duration_hours,unused_time_hours,charging_duration_hours
0,0,5e23b149f9af8b5fe4b973cf,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 17:31:35+00:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,194.0,"[{'WhPerMile': 250, 'kWhRequested': 25.0, 'mil...",6.039167,1.661111,4.378056
1,1,5e23b149f9af8b5fe4b973d0,2020-01-02 13:36:50+00:00,2020-01-02 22:38:21+00:00,2020-01-02 20:18:05+00:00,33.097,1_1_193_825_2020-01-02 13:36:49.599853,1,AG-1F01,1-1-193-825,America/Los_Angeles,4275.0,"[{'WhPerMile': 280, 'kWhRequested': 70.0, 'mil...",9.025278,2.337778,6.687500
2,2,5e23b149f9af8b5fe4b973d1,2020-01-02 13:56:35+00:00,2020-01-03 00:39:22+00:00,2020-01-02 16:35:06+00:00,6.521,1_1_193_829_2020-01-02 13:56:35.214993,1,AG-1F03,1-1-193-829,America/Los_Angeles,344.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile...",10.713056,8.071111,2.641944
3,3,5e23b149f9af8b5fe4b973d2,2020-01-02 13:59:58+00:00,2020-01-02 16:38:39+00:00,2020-01-02 15:18:45+00:00,2.355,1_1_193_820_2020-01-02 13:59:58.309319,1,AG-1F04,1-1-193-820,America/Los_Angeles,1117.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile...",2.644722,1.331667,1.313056
4,4,5e23b149f9af8b5fe4b973d3,2020-01-02 14:00:01+00:00,2020-01-02 22:08:40+00:00,2020-01-02 18:17:30+00:00,13.375,1_1_193_819_2020-01-02 14:00:00.779967,1,AG-1F06,1-1-193-819,America/Los_Angeles,334.0,"[{'WhPerMile': 400, 'kWhRequested': 16.0, 'mil...",8.144167,3.852778,4.291389


In [9]:
sessions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 66450 entries, 0 to 66449
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   Unnamed: 0               66450 non-null  int64              
 1   id                       66450 non-null  str                
 2   connectionTime           66450 non-null  datetime64[us, UTC]
 3   disconnectTime           66450 non-null  datetime64[us, UTC]
 4   doneChargingTime         62362 non-null  datetime64[us, UTC]
 5   kWhDelivered             66450 non-null  float64            
 6   sessionID                66450 non-null  str                
 7   siteID                   66450 non-null  int64              
 8   spaceID                  66450 non-null  str                
 9   stationID                66450 non-null  str                
 10  timezone                 66450 non-null  str                
 11  userID                   49187 non-null

In [10]:
sessions_df = sessions_df.drop(columns=sessions_df.filter(like="Unnamed").columns)
sessions_df = sessions_df.drop_duplicates()

In [11]:
sessions_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 65037 entries, 0 to 65036
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   id                       65037 non-null  str                
 1   connectionTime           65037 non-null  datetime64[us, UTC]
 2   disconnectTime           65037 non-null  datetime64[us, UTC]
 3   doneChargingTime         60950 non-null  datetime64[us, UTC]
 4   kWhDelivered             65037 non-null  float64            
 5   sessionID                65037 non-null  str                
 6   siteID                   65037 non-null  int64              
 7   spaceID                  65037 non-null  str                
 8   stationID                65037 non-null  str                
 9   timezone                 65037 non-null  str                
 10  userID                   47822 non-null  float64            
 11  userInputs               47822 non-null

In [13]:
import numpy as np

df = sessions_df[["spaceID", "connectionTime", "disconnectTime"]].copy()
df["connectionTime"] = pd.to_datetime(df["connectionTime"], utc=True).dt.tz_localize(None)
df["disconnectTime"] = pd.to_datetime(df["disconnectTime"], utc=True).dt.tz_localize(None)
df = df.dropna().sort_values(["spaceID", "connectionTime"])

rows = []
for space, g in df.groupby("spaceID", sort=False):
    start = g["connectionTime"].to_numpy(dtype="datetime64[ns]")
    end = g["disconnectTime"].to_numpy(dtype="datetime64[ns]")

    # Für jede Session: erster Index, dessen Start >= ihrem Ende ist
    upper = np.searchsorted(start, end, side="left")

    # Alle Sessions zwischen i+1 und upper[i]-1 starten vor Ende von i -> Überlappung
    for i in np.nonzero(upper > np.arange(len(g)) + 1)[0]:
        for j in range(i + 1, upper[i]):
            rows.append((space, start[i], end[i], start[j], end[j]))

overlaps = pd.DataFrame(rows, columns=["spaceID", "connectionTime_1", "disconnectTime_1",
                                       "connectionTime_2", "disconnectTime_2"])
print(f"{len(overlaps)} Überlappungen")
overlaps

1 Überlappungen


,spaceID,connectionTime_1,disconnectTime_1,connectionTime_2,disconnectTime_2
0,AG-3F20,2021-07-19 20:11:16,2021-07-19 21:58:08,2021-07-19 20:11:16,2021-07-20 00:25:14


In [15]:
import numpy as np

df = sessions_df[["spaceID", "connectionTime", "disconnectTime"]].copy()
df["connectionTime"] = pd.to_datetime(df["connectionTime"], utc=True).dt.tz_localize(None)
df["disconnectTime"] = pd.to_datetime(df["disconnectTime"], utc=True).dt.tz_localize(None)
df = df.dropna().sort_values(["spaceID", "connectionTime"])

rows = []
for space, g in df.groupby("spaceID", sort=False):
    idx = g.index.to_numpy()
    start = g["connectionTime"].to_numpy(dtype="datetime64[ns]")
    end = g["disconnectTime"].to_numpy(dtype="datetime64[ns]")
    upper = np.searchsorted(start, end, side="left")
    for i in np.nonzero(upper > np.arange(len(g)) + 1)[0]:
        for j in range(i + 1, upper[i]):
            rows.append((space, idx[i], idx[j]))

overlaps = pd.DataFrame(rows, columns=["spaceID", "index_1", "index_2"])
print(f"{len(overlaps)} Überlappungen")

1 Überlappungen


In [16]:
i1, i2 = overlaps["index_1"], overlaps["index_2"]

# Werte vor dem Tausch merken
disc_1 = sessions_df.loc[i1, "disconnectTime"].tolist()
conn_2 = sessions_df.loc[i2, "connectionTime"].tolist()

# Tausch in sessions_df
sessions_df.loc[i1, "disconnectTime"] = conn_2
sessions_df.loc[i2, "connectionTime"] = disc_1

In [17]:

df = sessions_df[["spaceID", "connectionTime", "disconnectTime"]].copy()
df["connectionTime"] = pd.to_datetime(df["connectionTime"], utc=True).dt.tz_localize(None)
df["disconnectTime"] = pd.to_datetime(df["disconnectTime"], utc=True).dt.tz_localize(None)
df = df.dropna().sort_values(["spaceID", "connectionTime"])

rows = []
for space, g in df.groupby("spaceID", sort=False):
    start = g["connectionTime"].to_numpy(dtype="datetime64[ns]")
    end = g["disconnectTime"].to_numpy(dtype="datetime64[ns]")

    # Für jede Session: erster Index, dessen Start >= ihrem Ende ist
    upper = np.searchsorted(start, end, side="left")

    # Alle Sessions zwischen i+1 und upper[i]-1 starten vor Ende von i -> Überlappung
    for i in np.nonzero(upper > np.arange(len(g)) + 1)[0]:
        for j in range(i + 1, upper[i]):
            rows.append((space, start[i], end[i], start[j], end[j]))

overlaps = pd.DataFrame(rows, columns=["spaceID", "connectionTime_1", "disconnectTime_1",
                                       "connectionTime_2", "disconnectTime_2"])
print(f"{len(overlaps)} Überlappungen")
overlaps

0 Überlappungen


,spaceID,connectionTime_1,disconnectTime_1,connectionTime_2,disconnectTime_2
